# 1. Load the dataset

In [57]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from IPython.display import Markdown
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import ChatOpenAI

In [2]:
file_path = 'data/Hands_On_Machine_Learning_with_Scikit_Learn_and_TensorFlow.pdf'
loader = PyPDFLoader(file_path)
document = loader.load()

In [3]:
print(f'Document has : {len(document)} pages ')
print(f'type of whole document : {type(document)}')
print(f'type of one page: {type(document[0])}')
print(f'meta data of first page: {document[0].metadata}')

Document has : 570 pages 
type of whole document : <class 'list'>
type of one page: <class 'langchain_core.documents.base.Document'>
meta data of first page: {'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)', 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)', 'creationdate': '2017-10-31T15:40:29+00:00', 'author': 'Aurélien Géron', 'moddate': '2017-10-31T12:10:13-04:00', 'title': 'Hands-On Machine Learning with Scikit-Learn and TensorFlow', 'trapped': '/False', 'source': 'data/Hands_On_Machine_Learning_with_Scikit_Learn_and_TensorFlow.pdf', 'total_pages': 570, 'page': 0, 'page_label': 'Cover'}


## 1.2 split the dataset into chapter 1-7

In [4]:
chapter_starts = [
    (24, 'Chapter 1. The Machine Learning Landscape'),
    (54, 'Chapter 2. End-to-End Machine Learning Project'),
    (102, 'Chapter 3. Classification'),
    (128, 'Chapter 4. Training Models'),
    (168, 'Chapter 5. Support Vector Machines'),
    (190, 'Chapter 6. Decision Trees'),
    (204, 'Chapter 7. Ensemble Learning and Random Forests'),
]

def get_chapter(page_number):
    chapter = 'Front Matter'
    for start_page, title in chapter_starts:
        if page_number >= start_page:
            chapter = title
        else:
            break
    return chapter

filter_doc = []
for page in document:
    if page.metadata['page'] < 248:
        chapter = get_chapter(page.metadata['page'])
        # add chapter info to metadata
        page_label = page.metadata.get('page_label', page.metadata['page'] + 1)
        page.metadata['chapter'] = chapter
        page.metadata['metadata_label'] = f"{chapter} | page {page_label}"
        filter_doc.append(page)


In [5]:
len(filter_doc)
filter_doc[120].metadata

{'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)',
 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)',
 'creationdate': '2017-10-31T15:40:29+00:00',
 'author': 'Aurélien Géron',
 'moddate': '2017-10-31T12:10:13-04:00',
 'title': 'Hands-On Machine Learning with Scikit-Learn and TensorFlow',
 'trapped': '/False',
 'source': 'data/Hands_On_Machine_Learning_with_Scikit_Learn_and_TensorFlow.pdf',
 'total_pages': 570,
 'page': 120,
 'page_label': '99',
 'chapter': 'Chapter 3. Classification',
 'metadata_label': 'Chapter 3. Classification | page 99'}

# 2. Chunking the dataset into smaller pieces

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200,
    add_start_index=True
)
chunks = text_splitter.split_documents(filter_doc)

In [7]:
print(f'the length of chunks {len(chunks)}')
print(f'the type of chunks : {type(chunks)}')
print(f'the type of one chunk: {type(chunks[0])}')
print(f'there are total {sum([len(chunk.page_content.lower()) for chunk in chunks])} characters in all chunks')

the length of chunks 604
the type of chunks : <class 'list'>
the type of one chunk: <class 'langchain_core.documents.base.Document'>
there are total 492175 characters in all chunks


# 3. Generate embeddings for each chunk

In [8]:
load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

In [9]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(
    model='text-embedding-3-small'
)

# 4. store the embeddings in a vector database

In [10]:
# Create an in-memory vector store using the embedder `embeddings` we created earlier
vector_store = InMemoryVectorStore(embedding=embeddings)

In [11]:
# Add the chunks to the vector store and store the result in a variable called `document_ids`
document_ids= vector_store.add_documents(chunks)

In [13]:
print(f'the length of document_id is {len(document_ids)} which is the same size with chunks ')
print(f'type of document_id is : {type(document_ids)}')
print(f'type of one element is {type(document_ids[0])}')

the length of document_id is 604 which is the same size with chunks 
type of document_id is : <class 'list'>
type of one element is <class 'str'>


In [ ]:
# sample to get the metadata of first opage
one_doc = vector_store.get_by_ids(document_ids[:1])[0]
Markdown(one_doc.page_content)
one_doc.metadata

{'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)',
 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)',
 'creationdate': '2017-10-31T15:40:29+00:00',
 'author': 'Aurélien Géron',
 'moddate': '2017-10-31T12:10:13-04:00',
 'title': 'Hands-On Machine Learning with Scikit-Learn and TensorFlow',
 'trapped': '/False',
 'source': 'data/Hands_On_Machine_Learning_with_Scikit_Learn_and_TensorFlow.pdf',
 'total_pages': 570,
 'page': 0,
 'page_label': 'Cover',
 'chapter': 'Front Matter',
 'metadata_label': 'Front Matter | page Cover',
 'start_index': 0}

## 4.1 test the vector database by querying with a sample question

In [60]:
query = 'how to do feature engineering'
retrieved_docs  = vector_store.similarity_search(query=query,k=3)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})


# 5. Initialize the LLM and set up the retrieval-based question-answering system

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
# prompt design :
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are a machine learning expert and tutor.

        Your task is to answer the question using ONLY the provided context.

        Guidelines:
        - Use ONLY the information from the context
        - Do NOT use your own knowledge
        - If the answer is not in the context, say: "I don't know based on the provided document"
        - Be clear and concise
        - Explain concepts simply
        - Provide examples only if they are supported by the context

        """
    ),
    (
        "human",
        """
        Context:
        {context}

        Question:
        {question}

        Answer:
        """
    ),
])

In [84]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(model="gpt-4o-mini",temperature= 0 )


def format_docs(docs):
    return "\n\n".join(
        f"[{doc.metadata.get('metadata_label', 'Unknown source')}]\n{doc.page_content}"
        for doc in docs
    )


rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)


In [85]:
response = rag_chain.invoke("tell me about ai agent")
display(Markdown(response))


I don't know based on the provided document.